# Week 15 – Week 16: Neural Networks from Scratch & Model Deployment Basics

**Student:** Tanmay Sankulwar  
**PRN:** 24070521058  
**Course:** B.Tech Computer Science / Information Technology (5th Semester)  
**Repository:** [HACKOWEEK-SEM-V-2026-27](https://github.com/Tanmay-1607/HACKOWEEK-SEM-V-2026-27)  

---

## 📌 Syllabus Objectives (SIT-N Hack-o-Week 5th Semester):
1. **Model Deployment Basics:** Save and load trained machine learning models using `pickle` and `joblib`. Measure serialization latency, deserialization latency, file size differences, and verify prediction consistency.
2. **Perceptrons & Activation Functions:** Implement a single-layer perceptron, test on boolean logic gates (AND, OR, NAND, XOR) to demonstrate the linear separability constraint (Minsky & Papert, 1969). Implement activation functions and their mathematical derivatives (Sigmoid, ReLU, Leaky ReLU, Tanh).
3. **Loss Functions & Optimization Algorithms:** Formulate Binary Cross-Entropy (BCE) and Mean Squared Error (MSE). Implement Stochastic Gradient Descent (SGD), SGD with Momentum, and Adaptive Moment Estimation (Adam) from scratch.
4. **Deep Neural Network from Scratch in NumPy:** Build a modular Multi-Layer Perceptron (MLP) featuring forward propagation with activation caching, loss calculation, backpropagation with the chain rule, and parameter updates.

## 1. Imports & Environment Setup

In [ ]:
import os
import time
import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All libraries imported successfully!")

## 2. Dataset Loading & Preprocessing Pipeline

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'kaggle_student_performance.csv')
df = pd.read_csv(DATA_PATH)
df['Distinction'] = (df['CGPA'] >= 8.5).astype(int)

numeric_features = ['Age', 'StudyHoursPerWeek', 'AttendanceRate', 'MathScore', 'ReadingScore', 'WritingScore']
categorical_features = ['Gender', 'Department']

X = df[numeric_features + categorical_features]
y = df['Distinction'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Dataset Loaded: {df.shape[0]} samples, {X.shape[1]} features")
print(f"Class Distribution: Distinction={np.sum(y==1)}, Regular={np.sum(y==0)}")

## 3. Objective 1: Model Deployment Basics (Pickle vs Joblib Benchmark)

In [ ]:
# Build Scikit-Learn Production Pipeline
prod_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features)
    ])),
    ('model', MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42))
])

print("Training production MLP pipeline...")
prod_pipeline.fit(X_train, y_train)
acc = prod_pipeline.score(X_test, y_test)
print(f"Production Pipeline Test Accuracy: {acc * 100:.2f}%")

os.makedirs('../saved_models', exist_ok=True)
pkl_path = '../saved_models/student_model.pkl'
joblib_path = '../saved_models/student_model.joblib'

# 1. Benchmark Pickle
t0 = time.perf_counter()
with open(pkl_path, 'wb') as f:
    pickle.dump(prod_pipeline, f, protocol=pickle.HIGHEST_PROTOCOL)
pkl_save_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
with open(pkl_path, 'rb') as f:
    loaded_pkl = pickle.load(f)
pkl_load_ms = (time.perf_counter() - t0) * 1000
pkl_size_kb = os.path.getsize(pkl_path) / 1024

# 2. Benchmark Joblib
t0 = time.perf_counter()
joblib.dump(prod_pipeline, joblib_path, compress=3)
joblib_save_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
loaded_joblib = joblib.load(joblib_path)
joblib_load_ms = (time.perf_counter() - t0) * 1000
joblib_size_kb = os.path.getsize(joblib_path) / 1024

print("\n--- Model Serialization Benchmark Results ---")
print(f"Pickle: Save={pkl_save_ms:.2f}ms | Load={pkl_load_ms:.2f}ms | Size={pkl_size_kb:.2f} KB")
print(f"Joblib: Save={joblib_save_ms:.2f}ms | Load={joblib_load_ms:.2f}ms | Size={joblib_size_kb:.2f} KB")
print(f"Verification (Identical outputs): {np.array_equal(loaded_pkl.predict(X_test), loaded_joblib.predict(X_test))}")

## 4. Objective 2: Perceptron & Boolean Logic Gates (AND, OR, NAND, XOR)

In [ ]:
class SinglePerceptron:
    def __init__(self, lr=0.1, epochs=50):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0

    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0.0
        for epoch in range(1, self.epochs + 1):
            errors = 0
            for xi, target in zip(X, y):
                z = np.dot(xi, self.weights) + self.bias
                pred = 1 if z >= 0 else 0
                error = target - pred
                if error != 0:
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
                    errors += 1
            if errors == 0:
                return True, epoch
        return False, self.epochs

    def predict(self, X):
        return np.where((np.dot(X, self.weights) + self.bias) >= 0, 1, 0)

X_gate = np.array([[0,0], [0,1], [1,0], [1,1]])
gates = {
    'AND': np.array([0, 0, 0, 1]),
    'OR': np.array([0, 1, 1, 1]),
    'NAND': np.array([1, 1, 1, 0]),
    'XOR': np.array([0, 1, 1, 0])
}

for name, y_gate in gates.items():
    p = SinglePerceptron()
    conv, ep = p.fit(X_gate, y_gate)
    acc = np.mean(p.predict(X_gate) == y_gate) * 100
    status = f"Converged in {ep} epochs" if conv else "Failed to converge (Linear inseparability)"
    print(f"[{name} Gate] Accuracy: {acc:5.1f}% | Status: {status}")

## 5. Objective 2 (Cont.): Activation Functions & Analytical Derivatives

In [ ]:
z = np.linspace(-5, 5, 200)

# Formulations & Derivatives
sig = 1 / (1 + np.exp(-z))
sig_grad = sig * (1 - sig)

relu = np.maximum(0, z)
relu_grad = np.where(z > 0, 1.0, 0.0)

lrelu = np.where(z > 0, z, 0.1 * z)
lrelu_grad = np.where(z > 0, 1.0, 0.1)

tanh = np.tanh(z)
tanh_grad = 1 - tanh**2

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(z, sig, 'b-', label='Sigmoid σ(z)'); axes[0, 0].plot(z, sig_grad, 'g--', label="σ'(z)")
axes[0, 0].set_title("Sigmoid Activation"); axes[0, 0].grid(True); axes[0, 0].legend()

axes[0, 1].plot(z, relu, 'b-', label='ReLU'); axes[0, 1].plot(z, relu_grad, 'g--', label="ReLU'(z)")
axes[0, 1].set_title("Rectified Linear Unit (ReLU)"); axes[0, 1].grid(True); axes[0, 1].legend()

axes[1, 0].plot(z, lrelu, 'b-', label='Leaky ReLU'); axes[1, 0].plot(z, lrelu_grad, 'g--', label="Leaky ReLU'(z)")
axes[1, 0].set_title("Leaky ReLU (α=0.1)"); axes[1, 0].grid(True); axes[1, 0].legend()

axes[1, 1].plot(z, tanh, 'b-', label='Tanh'); axes[1, 1].plot(z, tanh_grad, 'g--', label="Tanh'(z)")
axes[1, 1].set_title("Hyperbolic Tangent (Tanh)"); axes[1, 1].grid(True); axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 6. Objective 3 & 4: Deep Neural Network from Scratch in NumPy (with SGD & Adam)

In [ ]:
class NumPyNeuralNetwork:
    def __init__(self, layers, activations, optimizer='adam', lr=0.01):
        self.layers = layers
        self.activations = activations
        self.optimizer = optimizer
        self.lr = lr
        self.weights = []
        self.biases = []
        
        np.random.seed(42)
        for i in range(len(layers) - 1):
            n_in, n_out = layers[i], layers[i+1]
            scale = np.sqrt(2.0 / n_in) if activations[i] in ['relu', 'leaky_relu'] else np.sqrt(1.0 / n_in)
            self.weights.append(np.random.randn(n_in, n_out) * scale)
            self.biases.append(np.zeros((1, n_out)))
            
        self.v_w = [np.zeros_like(w) for w in self.weights]
        self.v_b = [np.zeros_like(b) for b in self.biases]
        self.s_w = [np.zeros_like(w) for w in self.weights]
        self.s_b = [np.zeros_like(b) for b in self.biases]
        self.t = 0

    def _act(self, z, kind):
        if kind == 'relu': return np.maximum(0, z)
        if kind == 'leaky_relu': return np.where(z > 0, z, 0.01 * z)
        if kind == 'tanh': return np.tanh(z)
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

    def _act_grad(self, z, a, kind):
        if kind == 'relu': return np.where(z > 0, 1.0, 0.0)
        if kind == 'leaky_relu': return np.where(z > 0, 1.0, 0.01)
        if kind == 'tanh': return 1.0 - a**2
        return a * (1.0 - a)

    def forward(self, X):
        a = X
        a_cache = [a]
        z_cache = []
        for W, b, act in zip(self.weights, self.biases, self.activations):
            z = np.dot(a, W) + b
            a = self._act(z, act)
            z_cache.append(z)
            a_cache.append(a)
        return a, z_cache, a_cache

    def backward(self, y_true, z_cache, a_cache):
        m = y_true.shape[0]
        grad_W = [None] * len(self.weights)
        grad_b = [None] * len(self.biases)
        delta = a_cache[-1] - y_true
        
        for l in reversed(range(len(self.weights))):
            grad_W[l] = (1.0 / m) * np.dot(a_cache[l].T, delta)
            grad_b[l] = (1.0 / m) * np.sum(delta, axis=0, keepdims=True)
            if l > 0:
                delta = np.dot(delta, self.weights[l].T) * self._act_grad(z_cache[l-1], a_cache[l], self.activations[l-1])
        return grad_W, grad_b

    def update(self, grad_W, grad_b):
        self.t += 1
        for l in range(len(self.weights)):
            gw, gb = grad_W[l], grad_b[l]
            if self.optimizer == 'sgd':
                self.weights[l] -= self.lr * gw
                self.biases[l] -= self.lr * gb
            elif self.optimizer == 'adam':
                self.v_w[l] = 0.9 * self.v_w[l] + 0.1 * gw
                self.v_b[l] = 0.9 * self.v_b[l] + 0.1 * gb
                self.s_w[l] = 0.999 * self.s_w[l] + 0.001 * (gw**2)
                self.s_b[l] = 0.999 * self.s_b[l] + 0.001 * (gb**2)
                vw_corr = self.v_w[l] / (1 - 0.9**self.t)
                vb_corr = self.v_b[l] / (1 - 0.9**self.t)
                sw_corr = self.s_w[l] / (1 - 0.999**self.t)
                sb_corr = self.s_b[l] / (1 - 0.999**self.t)
                self.weights[l] -= self.lr * vw_corr / (np.sqrt(sw_corr) + 1e-8)
                self.biases[l] -= self.lr * vb_corr / (np.sqrt(sb_corr) + 1e-8)

    def fit(self, X, y, epochs=150, batch_size=32):
        y_2d = y.reshape(-1, 1)
        m = X.shape[0]
        loss_hist = []
        for ep in range(1, epochs + 1):
            idx = np.random.permutation(m)
            X_s, y_s = X[idx], y_2d[idx]
            for start in range(0, m, batch_size):
                xb, yb = X_s[start:start+batch_size], y_s[start:start+batch_size]
                preds, zc, ac = self.forward(xb)
                gw, gb = self.backward(yb, zc, ac)
                self.update(gw, gb)
            full_preds, _, _ = self.forward(X)
            bce = -np.mean(y_2d * np.log(np.clip(full_preds, 1e-12, 1-1e-12)) + (1-y_2d) * np.log(np.clip(1-full_preds, 1e-12, 1-1e-12)))
            loss_hist.append(bce)
        return loss_hist

    def predict(self, X):
        preds, _, _ = self.forward(X)
        return (preds >= 0.5).astype(int).flatten()

# Preprocess student features
prep = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(sparse_output=False), categorical_features)
])
X_train_proc = prep.fit_transform(X_train)
X_test_proc = prep.transform(X_test)

print("Training NumPy Neural Network with Adam Optimizer...")
nn = NumPyNeuralNetwork(layers=[X_train_proc.shape[1], 16, 8, 1], activations=['relu', 'relu', 'sigmoid'], optimizer='adam', lr=0.01)
loss_history = nn.fit(X_train_proc, y_train, epochs=120)
test_preds = nn.predict(X_test_proc)
acc = accuracy_score(y_test, test_preds)
print(f"NumPy Scratch Neural Network Test Accuracy: {acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=['Regular', 'Distinction']))

## 7. Optimizer Convergence Visualization (SGD vs Adam)

In [ ]:
nn_sgd = NumPyNeuralNetwork(layers=[X_train_proc.shape[1], 16, 8, 1], activations=['relu', 'relu', 'sigmoid'], optimizer='sgd', lr=0.05)
loss_sgd = nn_sgd.fit(X_train_proc, y_train, epochs=120)

plt.figure(figsize=(9, 5))
plt.plot(loss_sgd, label='SGD (lr=0.05)', color='#f59e0b', linewidth=2)
plt.plot(loss_history, label='Adam (lr=0.01)', color='#10b981', linewidth=2)
plt.title('Training Loss Convergence: SGD vs Adam Optimizer')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True)
plt.show()